<div style="text-align: center"><img src="https://telematica.usm.cl/wp-content/themes/telematica-web/assets/img/header-usm.png" height="75"></div>

# Tarea 4: Mensajería CoAP y MQTT
### Asignatura: Redes de Sensores - TEL329 - 2026
#### Docente y autor: Berioska Contreras Vargas - berioska.contreras@usm.cl

###### Este contenido es parte de la asignatura TEL329 de UTFSM. Las tareas son un complemento a las lecciones teóricas. El objetivo es simular y emular redes de sensores constreñidos.

## Protocolo CoAP

El protocolo CoAP (Constrained Application Protocol) es un protocolo de la capa de aplicación diseñado para ser ligero y propuesto por la Fuerza de Trabajo de Ingeniería de Internet (IETF) el año 2014. Utiliza UDP como protocolo de transporte subyacente, lo que reduce la sobrecarga en comparación con TCP. CoAP consiste de un modelo de comunicación cliente-servidor, semejante a HTTP, y admite métodos como GET, POST, PUT y DELETE para interactuar con recursos específicos. CoAP se integra con direccionamiento IPv6 y con el mecanismo de seguridad DTLS (Datagram Transport Layer Security) para proporcionar confidencialidad en las comunicaciones. 

La mayor parte del código que usaremos se encuentra en $(CONTIKI)/examples/coap/.

| Fuente | Referencia | 
| - | - |
| [ContikiNG Docs](https://docs.contiki-ng.org/en/develop/doc/programming/CoAP.html) | Contiki Next Generation Documentation - CoAP |
| [ContikiNG Repo](https://github.com/contiki-ng/contiki-ng) | Contiki-NG: The OS for Next Generation IoT Devices |
| [RFC 7252: 2014](https://www.rfc-editor.org/rfc/rfc7252) | The Constrained Application Protocol (CoAP) |

### Servidor CoAP de ContikiNG

El código $coap-example-server.c$ implementa un simple servidor CoAP con varios recursos. Declara recursos CoAP externos p.e. res_hello, res_mirror, etc. que están definidos en archivos separados (examples/coap/coap-example-server/resources). Estos recursos representan distintas funcionalidades o datos expuestos por el servidor.

La llamada de recursos es posible a través de la **activación de cada recurso** lo que permite resolver o mapear cada recurso a una ruta $URI$ específica p.e. test/hello. El código $coap-example-server.c$ está a la espera eventos p.e. presionar el botón, y así reaccionar a ellos con acciones como ejecutar una notificación o actualización de variable. Inicie el servidor CoaP/Erbium de ContikiNG según los Pasos 1 y 2. Verifique que el proceso tiene asociada una dirección IPv6 local y global disponible a través de un túnel serial p.e. tunslip.

In [ ]:
# Paso 1 - Servidor CoAP ContikiNG
$ cd contiki-ng/examples/coap/coap-example-server/coap-example-server.c
$ sudo make distclean
$ sudo make target=native
$ sudo /build/native/coap-examples-server.native

# Paso 2 - Resultado esperado
...
[INFO: Main      ] Tentative link-local IPv6 address: fe80::302:304:506:708
[INFO: Native    ] Added global IPv6 address fd00::302:304:506:708
[INFO: App       ] Starting Erbium Example Server
...

### Cliente CoAP de ContikiNG

El código $coap-example-client.c$ implementa el cliente CoAP que periódicamente envía peticiones al servidor Coap y que envía notificaciones cada vez que se presione un botón del dispositivo. Además, define un servidor Erbium específico p.e. SERVER_EP "coap://[fe80::302:304:506:708]". Utiliza un arreglo de rutas de servicio que contienen URIs de recursos del servidor. El intervalo de envío de notificaciones es fijo p.e. Toggle_Interval. Cuando el temporizador expira, envía un petición POST con el contenido $Toggle!$


En general, el cliente permite que un usuario envíe distintas peticiones de recuros hacia el servidor a través de un evento específico p.e. presionar un botón, para luego desplegar la respuesta recibida. Inicie el cliente CoaP de ContikiNG según los Pasos 3 a 6. Verifique que el código respuesta sea consistente con el método y recurso consultado.

| Fuente | Referencia | 
| - | - |
| [CoAP-CLI](https://www.npmjs.com/package/coap-cli) | CoAP-CLI built on node.js |

In [ ]:
# Paso 3 - Cliente Coap ContikiNG
$ cd contiki-ng/examples/coap/coap-example-client/coap-example-client.c
$ sudo make distclean
$ sudo make target=native
$ sudo /build/native/coap-examples-client.native

# Paso 4 - Resultado esperado
...
--Toggle timer--
coap://[fe80::302:304:506:708]:5683
...

# Paso 5 - Instalar utilitarios de CoAP
$ sudo apt-get install -y npm
$ sudo npm install coap-cli -g

# Paso 6 - Consultar recursos
$ coap get coap://[fe80::302:304:506:708]:5683/.well-known/core
(2.05)
$ coap get coap://[fe80::302:304:506:708]:5683/led
(4.04)
$ coap get coap://[fe80::302:304:506:708]:5683/led?color=green mode=off
(4.04)
...

## Protocolo MQTT

El protocolo MQTT (Message Queue Telemetry Transport) es un protocolo diseñado sobre la capa de transporte TCP/IP, que le permite una integración directa con el protocolo TLS (Transport Layer Security) para una comunicación cifrada entre dispositivos. MQTT fue desarrollado originalmente por IBM a finales de 1990, y en 2014 se transformó en estándard abierto OASIS. El protocolo MQTT es asincrónico y separa la mensajería de envío y recepción en tiempo y espacio, siendo escalable para entornos de red inestables.

La mayor parte del código que usaremos se encuentra en $(CONTIKI)/examples/mqtt-client

| Fuente | Referencia | 
| - | - |
| [MQTT IBM](https://developer.ibm.com/articles/iot-mqtt-why-good-for-iot/) | Michael Yuan - Getting to Know MQTT |
| [MQTT OASIS v3,1](https://docs.oasis-open.org/mqtt/mqtt/v3.1.1/mqtt-v3.1.1.html) | OASIS Standard MQTT v3.1 |
| [MQTT ContikiNG](https://docs.contiki-ng.org/en/develop/doc/tutorials/MQTT.html) | ContikiNG MQTT Documentation |
| [MQTT Mosquitto](https://mosquitto.org/) | An OpenSource MQTT Broker |

### Broker MQTT de ContikiNG

El componente central de MQTT es el Broker, este actúa como un intermediario entre los clientes que publican mensajes (publishers) y los clientes que se suscriben a ellos (subscribers). Recibe los mensajes publicados por los clientes, incluyendo el tema al que pertenecen y la "calidad de servicio" (QoS) deseada. Filtra los mensajes en función de las suscripciones de los clientes. Solo entrega los mensajes a los clientes que se han suscrito a los temas correspondientes. Mantiene una lista de suscripciones de los clientes, incluyendo los temas a los que se han suscrito. Además, puede implementar mecanismos de seguridad para autenticar a los clientes y autorizar el acceso a los temas.

Utilizaremos el servicio Broker del proyecto abierto Mosquitto según los Pasos 7 y 8. Verifique que el servicio está escuchando en el 

| Fuente | Referencia | 
| - | - |
| [Mosquitto](https://mosquitto.org/) | An OpenSource MQTT Broker |
| [ActiveMQ](https://activemq.apache.org/) | An OpenSource MQTT Broker by Apache |

In [ ]:
# Paso 7 - Instalar Mosquitto broker & clients
$ sudo apt-get install -y mosquitto mosquitto-clients
$ sudo vi /etc/mosquitto/mosquitto.conf
...
listener 1883
allow_anonymous true
...
$ sudo systemctl status|stop|start mosquitto

#Paso 8 - Iniciar broker y resultado esperado
$ mosquitto -v
...
1728432200: Opening ipv4 listen socket on port 1883.
1728432200: Opening ipv6 listen socket on port 1883.
1728432200: mosquitto version 2.0.11 running
...

### Suscripción/Publicación MQTT

El código $mqtt-client.c$  implementa un cliente MQTT para ContikiNG que se conecta a un broker MQTT (como Mosquitto o IBM Watson IoT Platform) y publica datos de sensores periódicamente. Además, define un servidor Broker específico p.e. SERVER_EP $MQTT\_CLIENT\_BROKER\_IP\_ADDR "fd00::1"$. Maneja eventos MQTT como conexión, desconexión, publicación y suscripción.  Publica datos del dispositivo (como la plataforma, la placa, el número de secuencia y el tiempo de actividad) en un tema MQTT. Se suscribe a un tema para recibir comandos y controlar los LEDs del dispositivo.

En general, el código proporciona un cliente MQTT básico para ContikiNG, además el código es modular y extensible, lo que permite adaptarlo a diferentes necesidades y plataformas. Inicie el suscriptor/publicador MQTT según los Pasos 9 a 11. Verifique que el proceso tiene asociada una dirección IPv6 local y global disponible a través de un túnel serial p.e. tunslip.


In [ ]:
# Paso 9 - Suscripción de tema general desde terminales
$ mosquitto_sub -h localhost -t "#"
...
Telematica
$ mosquitto_pub -h localhost -t "usm/cssj/delo/ictel/json" -m "Telematica"
...
1728432939: Received PUBLISH from auto-E17C575F-1512-7E3D-F0F8-E67834553271 (d0, q0, r0, m0, 'usm/cssj/delo/ictel/json', ... (10 bytes))
1728432939: Sending PUBLISH to auto-2A2AC4C8-1112-55E2-6ED9-A75678A4F29C (d0, q0, r0, m0, 'usm/cssj/delo/ictel/json', ... (10 bytes))

#Paso 10 - Cliente MQTT ContikiNG y resultado esperado
$ cd contiki-ng/examples/mqtt-client/mqtt-client.c
$ sudo make distclean
$ sudo make target=native
$ sudo ./mqtt-client.native
...
opened tun device ``/dev/tun0''
tun0: flags=4305<UP,POINTOPOINT,RUNNING,NOARP,MULTICAST>  mtu 1500
        inet 127.0.1.1  netmask 255.255.255.255  destination 127.0.1.1
        inet6 fd00::1  prefixlen 64  scopeid 0x0<global>
        inet6 fe80::293b:7f98:adc5:19c0  prefixlen 64  scopeid 0x20<link>
        unspec 00-00-00-00-00-00-00-00-00-00-00-00-00-00-00-00  txqueuelen 500  (UNSPEC)
        RX packets 0  bytes 0 (0.0 B)
        RX errors 0  dropped 0  overruns 0  frame 0
        TX packets 0  bytes 0 (0.0 B)
        TX errors 0  dropped 0 overruns 0  carrier 0  collisions 0

[INFO: Main      ] Starting Contiki-NG-develop/v4.9-70-g3695d415e-dirty
[INFO: Main      ] - Routing: RPL Lite
[INFO: Main      ] - Net: tun6
[INFO: Main      ] - MAC: nullmac
[INFO: Main      ] - 802.15.4 PANID: 0xabcd
[INFO: Main      ] - 802.15.4 Default channel: 26
[INFO: Main      ] Node ID: 1800
[INFO: Main      ] Link-layer address: 0102.0304.0506.0708
[INFO: Main      ] Tentative link-local IPv6 address: fe80::302:304:506:708
[INFO: Native    ] Added global IPv6 address fd00::302:304:506:708
MQTT Client Process

## Actividades 1, 2 y 3:

| N° | Actividad | 
| - | - |
| 1 | Implemente una pequeña red WSN simulada que integre los protocolos RPL Lite y Coap, considerando un nodo raíz y 5 nodos clientes que notifiquen sólo un recurso p.e. temperatura generada de manera aleatoria. Hint random_rand(). Presente su código adaptado y visualice sus resultados en el simulador Cooja o Terminal.  |
| 2 | Implemente una pequeña red WSN simulada que integre los protocolos RPL Lite y MQTT, considerando un nodo raíz y 5 nodos clientes que notifiquen sólo un tema p.e. temperatura generada de manera aleatoria. Hint random_rand(). Presente su código adaptado y visualice sus resultados en el simulador Cooja o Terminal.  |
| 3 | Realice una captura de tráfico de la mensajería generada por el protocolo CoaP y el protocolo MQTT de los puntos 1 y 2. Luego, extraiga los tiempos de respuesta de una muestra representativa de mensajes de cada uno. Presente los datos en una tabla que resuma el tiempo de respuesta de consulta de cada protocolo, analice sus resultados y explíquelos de manera sucinta. |

# ACTIVIDADES RESUELTAS

> Respuestas de Francisco Zúñiga. Código, logs, pcap y CSV en el repo `tel329`, carpeta `tarea4/` (`a1-coap/`, `a2-mqtt/`, `a4-azure/`, `evidencia/`). Todo número de este notebook sale de una corrida; lo que falta está marcado **[FALTA]** y detallado en `entrega/Pendientes_Tarea4.md`.

### Actividad 1 — Red WSN con RPL Lite y CoAP

#### Qué se construyó

Una red de seis Cooja motes sobre la pila de Contiki-NG que vimos en la Lección 8 (aplicación CoAP sobre UDP/uIP, IPv6 con RPL, adaptación 6LoWPAN y MAC CSMA sobre IEEE 802.15.4). El nodo 1 arranca como raíz del DODAG con `NETSTACK_ROUTING.root_start()` y es además el servidor CoAP: activa un único recurso, `sensores/temp`, cuyo handler de POST/PUT registra lo que llega y responde `2.04 Changed`. Los nodos 2 a 6 son los clientes: se unen al DODAG, esperan a que `NETSTACK_ROUTING.node_is_reachable()` sea verdadero, toman la dirección de la raíz con `get_root_ipaddr()` y cada 10 segundos envían un POST **confirmable** a `coap://[raíz]:5683/sensores/temp` con el texto `id=<nodo> seq=<k> temp=<xx.x>`, donde la temperatura sale de `random_rand()` (`150 + random_rand() % 300` décimas de grado, es decir uniforme en 15,0–44,9 °C). La topología (UDGM, alcance 50 m) deja a n2, n3 y n4 a 40 m de la raíz y a n5 y n6 a 80 m, de modo que estos dos solo llegan a través de n2 y n3: la red tiene clientes a uno y a dos saltos y RPL es el que arma el camino.

```
          n3 (0,40)          n6 (0,80)
            |                  |      n6 solo alcanza a n3 → 2 saltos
 n4 ---- [n1 raíz] ---- n2 ---- n5
(-40,0)   (0,0)      (40,0)  (80,0)   n5 solo alcanza a n2 → 2 saltos
```

El código adaptado está en `tarea4/a1-coap/` (`root-coap.c`, `cliente-coap.c`, `resources/res-temp.c`, `Makefile`, `project-conf.h`) y parte de los ejemplos `examples/coap/coap-example-server` y `coap-example-client` de Contiki-NG, con tres cambios de fondo. Primero, el servidor del ejemplo no era raíz RPL ni el cliente conocía al servidor más que por una dirección fija (`SERVER_EP "coap://[fe80::202:2:2:2]"`); acá el endpoint es la raíz que RPL ya conoce, así que no hay direcciones hardcodeadas y el mismo binario sirve para los cinco clientes. Segundo, en vez de `COAP_BLOCKING_REQUEST`, que deja al protothread clavado hasta la respuesta, se usa la API no bloqueante `coap_send_request()` de `coap-callback-api.h`: el callback recibe el estado `COAP_REQUEST_STATUS_RESPONSE` con el mensaje (o `TIMEOUT` si se agotaron las retransmisiones), y eso permite imprimir el envío y la respuesta como líneas separadas, que Cooja marca con el tiempo simulado. Tercero, el recurso lee el hop limit del datagrama IPv6 que todavía está en `uip_buf` cuando corre el handler (`UIP_IP_BUF->ttl`) y de ahí deduce los saltos (`64 − ttl + 1`, porque uIP sale con `UIP_TTL = 64` y cada reenvío descuenta uno); lo devuelve en el payload de la respuesta para que el cliente también lo registre.

El corazón del cliente (`cliente-coap.c`):

```c
coap_init_message(request, COAP_TYPE_CON, COAP_POST, 0);
coap_set_header_uri_path(request, URI_TEMP);          /* "sensores/temp" */
coap_set_header_content_format(request, TEXT_PLAIN);
coap_set_payload(request, (uint8_t *)payload, strlen(payload));

t_envio = clock_time();
if(coap_send_request(&req_state, &server_ep, request, respuesta_cb)) {
  pendiente = 1;
  LOG_INFO("TX id=%u seq=%" PRIu32 " mid=%u len=%u '%s'\n",
           node_id, seq, request->mid, (unsigned)strlen(payload), payload);
}
```

y el handler del recurso en la raíz (`resources/res-temp.c`):

```c
uint8_t ttl = UIP_IP_BUF->ttl;
unsigned saltos = (unsigned)(UIP_TTL - ttl) + 1;
...
LOG_INFO("REG %s hops=%u ttl=%u mid=%u total=%" PRIu32 "\n",
         ultima, saltos, ttl, request->mid, total_recibidas);
snprintf((char *)buffer, preferred_size, "ok hops=%u", saltos);
coap_set_payload(response, buffer, strlen((char *)buffer));
coap_set_status_code(response, CHANGED_2_04);
```

Se eligió que el cliente notifique con POST y no que la raíz observe (RFC 7641) porque el enunciado pide que los clientes *notifiquen*, y en el modelo petición/respuesta de CoAP (RFC 7252 §5; Lección 10, "Modelo de Petición/Respuesta") la forma directa de empujar un dato al servidor es una petición POST o PUT. Se eligió confirmable y no non-confirmable porque con CON el servidor responde con la respuesta *piggybacked* en el ACK (RFC 7252 §5.2.1) y esa ida y vuelta es exactamente la que mide la Actividad 3; con NON no habría respuesta que medir. El motor CoAP (proceso `coap_engine`, UDP 5683, RFC 7252 §6.1) lo levanta `contiki-main.c` al incluir el módulo `coap` en el Makefile, igual que en los ejemplos.

#### Qué se ve

Corrida headless con semilla 123456 (`a1-coap/correr.sh`, log completo en `tarea4/evidencia/a1-coap.log`). La raíz crea el DAG a los 0,38 s (`created DAG with instance ID 0, DAG ID fd00::201:1:1:1, rank 128`). Los clientes a un salto obtienen ruta y mandan su primer POST a los 20,7 s (n4), 22,7 s (n2) y 24,6 s (n3); los de dos saltos a los 36,9 s (n5) y 42,3 s (n6), porque primero tienen que oír un DIO de un nodo que ya tenga padre. Desde ahí cada ronda deja tres líneas, por ejemplo:

```
20695000 n4 [INFO: A1cli ] TX id=4 seq=1 mid=17114 len=20 'id=4 seq=1 temp=24.2'
20712624 n1 [INFO: A1raiz] REG id=4 seq=1 temp=24.2 hops=1 ttl=64 mid=17114 total=1
20719824 n4 [INFO: A1cli ] RX id=4 seq=1 code=2.04 mid=17114 rtt_ms=25 'ok hops=1'
```

y para n5 y n6 el mismo patrón con `hops=2 ttl=63`. En 282,4 s simulados se enviaron 129 POST (25 a 27 por cliente) y llegaron 129 respuestas `2.04 Changed`: ningún timeout, ninguna ronda saltada por petición en vuelo y ninguna retransmisión CoAP. El `.csc` para la GUI (`a1-coap.csc`) muestra lo mismo en el *Log Listener* filtrado por `A1`, el DODAG en el *Visualizer* y el tráfico en el *Timeline* y el *Radio Logger*.

**Evidencia:** `evidencia/a1-coap.log` (Mote output, 2467 líneas), `evidencia/a1-coap.pcap` (820 tramas 802.15.4), `evidencia/a1-tiempos-coap.csv` y `evidencia/a1-resumen-coap.md`. **[FALTA]** captura de pantalla de la GUI (Visualizer/Timeline/Radio Logger): la corrida se hizo headless; `a1-coap.csc` queda listo para abrirlo y sacarla.


### Actividad 2 — Red WSN con RPL Lite y MQTT

#### Qué se construyó

Se implementó en Cooja una WSN de seis motes: el mote 1 es el `rpl-border-router` de Contiki-NG sin modificar, raíz del DODAG, y los motes 2 a 6 corren `mqtt-cliente.c`, una adaptación de `examples/mqtt-client` que publica cada 5 s en un único tema, `usm/tel329/temp`, una temperatura aleatoria obtenida con `random_rand()` (entre 15,0 y 34,9 °C, en décimas para no usar punto flotante). El payload es JSON: `{"id":<mote>,"seq":<n>,"temp":<xx.x>,"t":<ms de reloj>}`. El broker es Mosquitto 2.0.11, que corre en el host (el contenedor Docker) y escucha en `fd00::1:1883`; entre la red 6LoWPAN y el host está `tunslip6`, conectado al puerto serial del border router a través del *Serial Socket Server* de Cooja (puerto 60001), que crea la interfaz `tun0` con el prefijo `fd00::/64` y se lo anuncia al border router.

```
mote 2..6 (mqtt-cliente)  ──RPL Lite / 6LoWPAN──▶  mote 1 (border router)
                                                        │ serial (SerialSocketServer :60001)
                                                   tunslip6  ──▶ tun0 = fd00::1/64
                                                        │
                                                   mosquitto :1883   ◀── tcpdump
```

Del ejemplo se conservó la máquina de estados (registro → esperar ruta por defecto → `mqtt_connect` → publicar) y se quitó lo que no aportaba: IBM Watson, LEDs, botón, extensiones por plataforma, suscripción y el *ping* al padre. Lo que se cambió a propósito: el cliente usa **MQTT 3.1.1** (`MQTT_CONF_VERSION`; Contiki usa 3.1 por defecto) porque es el estándar OASIS que cita el enunciado; publica con **QoS 1** para recibir un `PUBACK` por cada mensaje, que es lo que permite medir ida y vuelta en la Actividad 3; el `client id` es `tel329-a2-mote<N>`; y el tamaño máximo de segmento TCP se subió de 32 a **64 bytes**, de modo que un `PUBLISH` completo (59 a 61 bytes: cabecera fija, tema, *packet id* y payload) viaje en un solo segmento y en una sola trama 802.15.4 sin fragmentación 6LoWPAN. El *keep alive* queda en 15 s (tres periodos de publicación, como en el ejemplo). Cada cliente imprime en el *Mote output* una línea `A2,PUB,…` al llamar a `mqtt_publish()` y una `A2,PUBACK,…` al recibir el evento `MQTT_EVENT_PUBACK`, con el `packet id` para emparejarlas, más el *rank* y el padre RPL del momento.

La configuración que define el experimento (`a2-mqtt/project-conf.h`):

```c
#define UIP_CONF_TCP 1                                   /* MQTT va sobre TCP; uIP lo trae apagado */
#define MQTT_CONF_VERSION MQTT_PROTOCOL_VERSION_3_1_1    /* el estándar OASIS que cita el enunciado */
#define A2_BROKER_IP   "fd00::1"                         /* el host del otro lado de tunslip6 */
#define A2_BROKER_PORT 1883
#define A2_TOPIC       "usm/tel329/temp"
#define A2_INTERVALO_S 5
#define A2_KEEP_ALIVE_S (3 * A2_INTERVALO_S)
#define A2_MAX_TCP_SEGMENT 64                            /* un PUBLISH completo en un segmento y una trama */
```

y la publicación en `mqtt-cliente.c`:

```c
temp_x10 = 150 + (random_rand() % 200);                  /* [15.0, 34.9] C en decimas */
len = snprintf(app_buffer, APP_BUFFER_SIZE,
               "{\"id\":%u,\"seq\":%u,\"temp\":%u.%u,\"t\":%lu}",
               node_id, seq_nr_value, temp_x10 / 10, temp_x10 % 10,
               (unsigned long)clock_time());
st = mqtt_publish(&conn, &mid, pub_topic, (uint8_t *)app_buffer, len,
                  MQTT_QOS_LEVEL_1, MQTT_RETAIN_OFF);
printf("A2,PUB,%u,%u,%u,%lu,%u,%u\n", node_id, seq_nr_value, mid,
       (unsigned long)clock_time(), rank_dag(), padre_id());
/* ... y en mqtt_event(): */
case MQTT_EVENT_PUBACK:
  printf("A2,PUBACK,%u,%u,%lu\n", node_id, *((uint16_t *)data),
         (unsigned long)clock_time());
```

La topología se eligió para que la red tenga profundidad y no todos los clientes estén a un salto: con alcance UDGM de 50 m, los motes 2 y 3 están a 30 m de la raíz (1 salto), 4 y 5 a 60 m (2 saltos, vía 2 y 3) y 6 a 90 m (3 saltos, vía 4). RPL Lite, en modo *non-storing*, formó exactamente ese DODAG: la raíz aprende las rutas descendentes con los DAO y encamina hacia abajo con cabecera de ruteo por fuente (SRH), cosa que se ve en el log (`read SRH, path len 1`).

Todo corre con un solo `docker run --privileged` de la imagen oficial `contiker/contiki-ng` y el script `a2-mqtt/correr.sh`: arranca Mosquitto, lanza Cooja sin GUI con el `.csc`, espera a que el socket serial del mote 1 esté abierto, lanza `tunslip6`, captura con `tcpdump` en `tun0` y al terminar vuelca los logs y calcula los tiempos. La simulación corre a velocidad 1,0 (tiempo real) porque el broker y `tunslip6` viven en el reloj del host.

#### Qué se ve

En `a2-mosquitto.log` los cinco clientes se conectan (`New client connected from fd00::20N:N:N:N as tel329-a2-mote<N> (p2, c1, k15)`: protocolo 3.1.1, *clean session*, *keep alive* 15) y a partir de ahí cada 5 s aparece el par `Received PUBLISH from tel329-a2-mote<N> (d0, q1, r0, m<mid>, 'usm/tel329/temp', ... (38 bytes))` / `Sending PUBACK`. En `a2-mqtt.log` (el *Mote output* con tiempo simulado en µs) están las líneas `A2,PUB` y `A2,PUBACK` de cada cliente, junto con la formación del DODAG (DIO, DAO, DAO-ACK). En `a2-tunslip6.log` está la consola del border router (`Routing: RPL Lite`, el prefijo recibido, su dirección global). Y en `a2-mqtt.pcap` está el tráfico TCP/MQTT tal como llega al host: los *hop limit* con que llegan los paquetes (63, 62 y 61) confirman los 1, 2 y 3 saltos de la topología.

| cliente | START (s) | primer CONNECTED (s) | primer PUB (s) | desconexiones |
|---|---|---|---|---|
| 2 | 0,2 | 22,0 | 22,5 | 0 |
| 3 | 0,9 | 12,3 | 12,4 | 0 |
| 4 | 0,3 | 34,7 | 35,1 | 0 |
| 5 | 0,7 | 24,8 | 25,2 | 0 |
| 6 | 0,2 | 47,7 | 48,0 | 0 |

| origen (mote) | paquetes hacia el broker | hop limit visto en tun0 | saltos = 64 − hlim |
|---|---|---|---|
| fd00::202:2:2:2 | 121 | 63 | 1 |
| fd00::203:3:3:3 | 122 | 63 | 1 |
| fd00::204:4:4:4 | 114 | 62 | 2 |
| fd00::205:5:5:5 | 116 | 62 | 2 |
| fd00::206:6:6:6 | 109 | 61 | 3 |

En 299 s simulados: 273 `PUBLISH` QoS 1 y 273 `PUBACK` (entre 51 y 58 por cliente), 0 desconexiones, 0 `PINGREQ`/`PINGRESP`.

**Evidencia:** `evidencia/a2-mqtt.log`, `a2-mosquitto.log`, `a2-tunslip6.log`, `a2-mqtt.pcap` y su volcado `a2-mqtt-pcap.txt` (901 paquetes TCP al/desde :1883), `a2-tiempos-mqtt.csv`, `a2-resumen-mqtt.md`. **[FALTA]** captura de pantalla de la GUI (`a2-mqtt.csc` abre con el *Serial Socket* ya en el puerto 60001 y el *Log Listener* filtrado por `A2,`).


### Actividad 3 — Captura de tráfico y tiempos de respuesta: CoAP vs MQTT

#### Cómo se capturó

Las dos mitades usan la misma fuente primaria: el *Mote output* de Cooja, que antepone a cada línea el instante simulado en microsegundos. En CoAP el tiempo de respuesta de cada petición es la diferencia entre la línea `RX` y la línea `TX` del mismo par (cliente, seq) — del `POST` confirmable a su respuesta `2.04` piggybacked en el ACK — y la empareja `a1-coap/tiempos.py` (→ `evidencia/a1-tiempos-coap.csv`, una fila por petición). En MQTT es la diferencia entre `A2,PUBACK` y `A2,PUB` del mismo `packet id` — de `mqtt_publish()` al evento `MQTT_EVENT_PUBACK` — y la empareja `a2-mqtt/analizar.py` (→ `evidencia/a2-tiempos-mqtt.csv`). Los saltos salen en CoAP del hop limit que lee la raíz (`64 − ttl + 1`) y en MQTT de la cadena de padres RPL que reporta cada cliente, cotejada con el hop limit del pcap en `tun0`.

Las capturas de tráfico son distintas por naturaleza del camino. En CoAP el tráfico nunca sale de la red de radio, así que la captura es del radio: `evidencia/a1-coap.pcap`, 820 tramas IEEE 802.15.4 (linktype 195), de 4,28 s a 282,36 s. Cooja trae esa exportación en el *Radio Logger* ("6LoWPAN Analyzer with PCAP"), pero ese plugin es un `VisPlugin` y la versión actual no lo arranca en `--no-gui`, así que `js/volcar-log.js` escribe el pcap por su cuenta enganchándose a los triggers de transmisión del radio medium, con el mismo formato que `PcapExporter.java`. En MQTT el tráfico cruza al host, y ahí se capturó con `tcpdump` en `tun0`: `evidencia/a2-mqtt.pcap` (901 paquetes TCP al/desde :1883, reloj del host). Las tramas de Cooja no llevan FCS, así que para abrir `a1-coap.pcap` en Wireshark hay que fijar *IEEE 802.15.4 → FCS format: None* (`-o wpan.fcs_format:0` en `tshark`). Los dos pcap se validaron con `tshark` 3.6.2 (`evidencia/a1-tshark.txt`, `evidencia/a2-tshark.txt`): en `a1-coap.pcap` decodifica 6LoWPAN y CoAP en 358 de las 820 tramas, 179 `CON POST /sensores/temp` y 179 `ACK 2.04 Changed` con 129 MID distintos de cada tipo, que son las 129 peticiones del log (las 50 tramas de más por tipo son las peticiones a dos saltos, que el radio ve dos veces con el mismo MID: la del cliente y el reenvío del mote intermedio); en `a2-mqtt.pcap` decodifica MQTT en 556 de los 901 paquetes, 5 `CONNECT`, 5 `CONNACK`, 273 `PUBLISH` y 273 `PUBACK`, sin `PINGREQ` ni retransmisiones de `PUBLISH`, lo mismo que cuenta el log de los motes.

#### Tabla comparativa

Tiempo de respuesta por mensaje visto desde la aplicación del sensor, en ms de tiempo simulado, agrupado por saltos de radio hasta la raíz. CoAP: 129 `POST` CON → `2.04`, 0 timeouts, 0 retransmisiones (corrida de 282,4 s). MQTT: 273 `PUBLISH` QoS 1 → `PUBACK`, 0 sin `PUBACK` (corrida de 299 s). Ambas con semilla 123456, UDGM con éxito 1,0, MAC CSMA, RPL Lite *non-storing*.

| saltos | protocolo | n | media | mediana | p95 | mín | máx |
|---|---|---|---|---|---|---|---|
| 1 | CoAP (POST CON → 2.04) | 79 | 48,4 | 46,8 | 81,8 | 13,5 | 90,8 |
| 1 | MQTT (PUBLISH QoS 1 → PUBACK) | 114 | 114,3 | 60,3 | 97,7 | 19,3 | 6238,3 |
| 2 | CoAP | 50 | 97,0 | 98,4 | 138,4 | 44,4 | 141,4 |
| 2 | MQTT | 108 | 168,4 | 164,3 | 246,2 | 67,6 | 284,6 |
| 3 | CoAP | — | — | — | — | — | — |
| 3 | MQTT | 51 | 213,2 | 211,8 | 290,8 | 132,8 | 381,8 |
| **todos** | **CoAP** | **129** | **67,2** | **61,8** | **121,4** | **13,5** | **141,4** |
| **todos** | **MQTT** | **273** | **154,2** | **130,6** | **243,7** | **19,3** | **6238,3** |

Notas de la tabla. Las filas por salto de CoAP se agregan desde `a1-tiempos-coap.csv` (n2+n3+n4 a un salto, n5+n6 a dos) con el mismo `p95` de `tiempos.py` (rango más cercano); las de MQTT son las de `a2-resumen-mqtt.md`, cuyo `analizar.py` interpola el p95. La red CoAP no tiene clientes a tres saltos. La fila MQTT de un salto está arrastrada por una sola muestra de 6238,3 ms (la primera publicación del mote 2); sin ella la media del mote 2 es indistinguible de la del mote 3 (61,5 ms). El detalle por cliente está en `evidencia/a1-resumen-coap.md` y `evidencia/a2-resumen-mqtt.md`.

#### Análisis

**Primero la salvedad: la comparación mezcla protocolo y camino.** En la Actividad 1 el servidor CoAP corre en la propia raíz del DODAG, así que la respuesta `2.04` sale del mismo mote que termina la ruta RPL y la ida y vuelta es puramente de radio. En la Actividad 2 el broker está en el host: el `PUBACK` nace en Mosquitto después de cruzar el border router, el enlace serial SLIP (`tunslip6`), `tun0` y la pila TCP de Linux, y vuelve por el mismo camino. Entonces la diferencia entre filas no es "CoAP contra MQTT" a secas, es "CoAP sobre UDP terminado en la raíz" contra "MQTT sobre TCP terminado en un broker detrás de un border router". Separar las dos contribuciones exigiría poner el servidor CoAP también detrás de `tunslip6`; **[FALTA, opcional]** no se hizo. Con esa salvedad, lo que las cifras sí dicen es lo siguiente.

**A igual número de saltos las medianas son comparables y el orden es el esperado.** A un salto, 46,8 ms CoAP contra 60,3 ms MQTT; a dos, 98,4 contra 164,3 ms. Los dos protocolos hacen una ida y una vuelta por dato (RFC 7252 §5.2.1: respuesta piggybacked en el ACK del CON; OASIS MQTT 3.1.1 §3.4: `PUBACK` por cada `PUBLISH` con QoS 1), así que la mediana la fija el costo por trama, no el protocolo de aplicación. La diferencia que sobra la explican tres cosas: MQTT carga 20 bytes de cabecera TCP por segmento donde CoAP carga 8 de UDP y 4 de cabecera CoAP, lo que vuelve las tramas más largas; en los primeros intercambios de cada conexión Linux responde con dos paquetes por publicación (el ACK puro y el segmento con el `PUBACK`, separados mientras está en *quick ACK*), que son dos tramas más bajando por la red de radio; y el `PUBACK` paga además el tramo SLIP + host que el `2.04` de CoAP no paga.

**Lo que domina en las dos es el backoff CSMA por trama, no el aire ni la aplicación.** Una `POST` de CoAP ocupa 81–90 bytes de trama, que a 250 kb/s son unos 2,7–2,9 ms de aire, más el ACK de enlace de 5 bytes y los turnarounds; la respuesta, 56–64 bytes, unos 2 ms. Si solo contara el aire, un RTT a un salto rondaría los 6 ms, y el mínimo medido (13,5 ms) está cerca de eso. Lo que abre el rango hasta 45 ms por sentido es el backoff que el CSMA de Contiki-NG aplica **antes de cada transmisión**, aunque el canal esté libre: `csma-output.c` retrasa cada trama nueva un tiempo uniforme en `[0, (2^BE − 1) · backoff_period)` con `CSMA_MIN_BE = 3`, y en la plataforma Cooja `backoff_period()` devuelve 6 ms, es decir un retardo uniforme en [0, 42) ms. Sumado a los ~3 ms de aire da el intervalo 5,6–45,6 ms que se midió por sentido en CoAP, y la distribución casi plana de los RTT a un salto es la suma de dos uniformes. Por eso cada salto de radio suma del orden de 50 ms de ida y vuelta en los dos protocolos: en CoAP la mediana pasa de 46,8 a 98,4 ms de uno a dos saltos; en MQTT de 60,3 a 164,3 y a 211,8 ms de uno a dos a tres. Cada salto agrega una trama hacia arriba y otra hacia abajo, con su backoff, su ACK de enlace y el reenvío en el mote intermedio; la bajada además lleva la cabecera de ruteo por fuente que pone la raíz en RPL Lite *non-storing*. El MAC es CSMA, no ContikiMAC ni TSCH: los radios están siempre encendidos, así que no hay latencia por ciclo de trabajo.

**La cola es donde los protocolos se separan de verdad: el outlier de 6,2 s.** El máximo de CoAP en toda la corrida es 141,4 ms; el de MQTT, 6238,3 ms, la primera publicación del mote 2. Esa muestra es una retransmisión TCP. El enlace SLIP es un cuello de botella serial sin control de flujo: correlacionando los `tcp: bad checksum` del log de los motes (tres en toda la corrida: dos en el mote 2, uno en el 4) con la captura en `tun0`, cada uno coincide con un par de paquetes que el broker emitió hacia el mismo mote con menos de 1 ms de separación (ACK puro seguido del segmento con `CONNACK` o `PUBACK`); el segundo llega corrupto al mote y uIP lo descarta. El dato no se pierde, TCP lo repara, pero lo repara con su RTO, que para esa conexión quedó inflado por el *handshake* lento. Es un costo estructural de confiar la entrega a TCP: la recuperación es correcta pero la latencia la decide el temporizador de retransmisión, no la red. CoAP también retransmite un `CON` si no llega el ACK en `ACK_TIMEOUT · ACK_RANDOM_FACTOR` (RFC 7252 §4.2: 2 s × 1,5; Contiki-NG usa `COAP_RESPONSE_TIMEOUT` = 3 s), hasta `MAX_RETRANSMIT` = 4 veces, y una sola retransmisión habría puesto su RTT también en el orden de los segundos; pero en esta corrida no hubo ninguna (0 `Retransmitting` en el log, los 129 `mid` respondidos al primer intento), porque el medio UDGM no pierde tramas y las colisiones entre clientes las absorbe la retransmisión de enlace de CSMA, que es decenas de ms. La diferencia de fondo es que los reintentos de CoAP los controla la aplicación con tiempos acotados y cada datagrama es independiente, mientras que en MQTT una pérdida frena la cola TCP entera. En una red real con pérdidas, la cola de los dos la darían las retransmisiones, pero las de MQTT las fija el RTO de TCP y las de CoAP el temporizador de la aplicación.

**Lo que TCP cobra antes del primer dato: el handshake bloqueado hasta el DAO.** MQTT exige una conexión TCP (OASIS MQTT 3.1.1 §4.2) y después `CONNECT`/`CONNACK` antes de publicar. En la red RPL eso tiene un costo que no se ve en el RTT pero sí en el tiempo hasta la primera publicación: el cliente abre el TCP apenas tiene dirección global y ruta por defecto, pero el `SYN-ACK` del broker solo puede bajar cuando la raíz ya tiene la ruta descendente, es decir después del `DAO`/`DAO-ACK` del cliente. En la captura se ve al broker retransmitir el `SYN-ACK` (1, 2 y 4 s de *backoff*) hasta que la ruta existe; mientras tanto el mote reenvía el `SYN` (18 `SYN` y 27 `SYN-ACK` en la captura para solo 5 conexiones). Cada cliente recibió su `DAO-ACK` entre 0,7 y 4,6 s antes de lograr el `CONNECTED`, y la primera publicación llegó entre 12 s (mote 3) y 48 s (mote 6, a tres saltos) después del arranque. Con CoAP sobre UDP ese *handshake* no existe: el primer datagrama ya es el dato (los clientes CoAP mandaron su primer `POST` apenas tuvieron ruta, entre 20,7 y 42,3 s, tiempo que es íntegramente de convergencia RPL), y si no hubiera ruta de vuelta solo se perdería la respuesta de ese mensaje, no se postergaría toda la sesión. Tampoco hubo *keep alive* que pagar: con 15 s de *keep alive* y publicaciones cada 5 s no apareció ningún `PINGREQ`/`PINGRESP`; si el sensor publicara menos seguido que su *keep alive*, MQTT pagaría un par de paquetes por intervalo solo para mantener la conexión (§3.1.2.10), costo que CoAP no tiene porque no mantiene conexión.

**El tamaño del mensaje casi no pesa, y ninguno fragmenta.** El payload CoAP son 20–21 bytes de texto; con la cabecera CoAP de 4 bytes (RFC 7252 §3; Lección 10, "Mensaje CoAP"), las opciones Uri-Path y Content-Format, y UDP e IPv6 comprimidos por 6LoWPAN IPHC, las `POST` ocupan 81–90 bytes de trama y las respuestas 56–64; el `PUBLISH` MQTT completo pesa 59–61 bytes (cabecera fija, tema, *packet id* y payload JSON de 38 bytes) y se forzó un segmento TCP de 64 bytes justamente para que fuera en una sola trama. Todo cabe en los 127 bytes de 802.15.4, así que en ninguno de los dos hay fragmentación 6LoWPAN, que habría agregado una trama más por sentido y por salto.

**En resumen.** En esta red el tiempo de respuesta de CoAP es, en mediana, 47 ms a un salto y ~98 ms a dos; el de MQTT con QoS 1, 60 ms a un salto, 164 a dos y 212 a tres, entre 1,3 y 1,7 veces el de CoAP a igual profundidad, con la salvedad de que el `PUBACK` cruza además el border router y el host. Los dos escalan linealmente con los saltos porque lo que manda es el backoff CSMA por trama (hasta 42 ms por trama en Cooja), no la cabecera de aplicación. La diferencia cualitativa está en la cola: CoAP sobre UDP no tiene *handshake*, no mantiene conexión y sus reintentos son acotados por la aplicación, de modo que su máximo quedó en 141 ms; MQTT sobre TCP paga el *handshake* bloqueado hasta el DAO, el *keep alive* si publica poco, y cuando el enlace SLIP corrompe un segmento la latencia la decide el RTO de TCP (6,2 s). Lo que MQTT da a cambio es la entrega *al menos una vez* con *packet id* (§4.3.2), el desacople publicador/suscriptor por el broker y la sesión persistente, que CoAP no ofrece por sí solo.

**Supuestos y límites.** Red simulada (UDGM con éxito 1,0 y sin pérdida por distancia, 6 motes, semilla fija 123456), broker en la misma máquina: los números miden el costo del protocolo y de la pila, no un enlace real. En MQTT el tiempo simulado corre a 1,0× contra el reloj del host; en los tramos en que el host se atrasa, el RTT medido desde el mote incluye ese atraso. Los conteos de tramas y paquetes de los dos pcap coinciden con los logs: `tshark` decodifica las 129 peticiones CoAP con su `2.04` y los 273 pares `PUBLISH`/`PUBACK` (ver *Cómo se capturó*).


## Sensores en la Nube

La conexión de los sensores WSN a la nube computacional ofrece una serie de beneficios que van desde el almacenamiento y procesamiento de datos hasta la gestión remota y el análisis avanzado. Esto permite desarrollar aplicaciones más sofisticadas, eficientes y escalables en una variedad de campos. Por ejemplo: Agricultura de precisión, Monitorización ambiental, Ciudades inteligentes, entre otros.

Los servicios en la nube se pueden clasificar en tres grandes categorías:  Infraestructura como servicio (IaaS), Plataforma como servicio (PaaS) y  Software como servicio (SaaS). 

IaaS proporciona acceso a recursos informáticos fundamentales como servidores, almacenamiento, redes y sistemas operativos a través de Internet. Ejemplos: Amazon Web Services (AWS) EC2, Microsoft Azure Virtual Machines, Google Compute Engine.

PaaS proporciona un entorno de desarrollo e implementación de aplicaciones en la nube. Ejemplos: AWS Elastic Beanstalk, Google App Engine, Microsoft Azure App Service.

SaaS proporciona acceso a aplicaciones de software a través de Internet. Ejemplos: Gmail, Salesforce, Dropbox, Google Workspace.

| Fuente | Referencia | 
| - | - |
| [NIST-SP-800-145](https://www.nist.gov/publications/nist-definition-cloud-computing) | Peter M. Mell and Timothy Grance. (2011). The NIST definition of cloud computing. National Institute of Standards and Technology. |
| [NIST-IR-8200](https://nvlpubs.nist.gov/nistpubs/ir/2018/NIST.IR.8200.pdf) | Editors Mike Hogan and Ben Piccarreta. (2018). Interagency Report on the Status of International Cybersecurity Standardization for the Internet of Things (IoT).|

## Microsoft IoT Hub

Microsoft IoT Hub, también conocido como Azure IoT Hub, es un servicio administrado en la nube que actúa como un centro de mensajes para la comunicación entre una aplicación de Internet de las Cosas (IoT) y los dispositivos conectados. IoT Hub permite conectar millones de dispositivos de manera segura y confiable, facilitancdo la comunicación tanto desde los dispositivos hacia la nube (telemetría) como desde la nube hacia los dispositivos (comandos y notificaciones). Puede manejar millones de dispositivos y eventos por segundo, lo que lo hace adecuado para grandes implementaciones de IoT.

Nuestro objetivo es conectar sensores de Contiki-NG en la plataforma Azure IoT Hub. 

| Fuente | Referencia | 
| - | - |
| [Azure IoT Hub](https://learn.microsoft.com/es-es/azure/iot-hub/iot-concepts-and-iot-hub) | ¿Qué es Microsoft IoT Hub? |

## Actividad 4

El objetivo es desarrollar o implementar las siguientes aplicaciones:

| Aplicación | Descripción | 
| - | - |
| Sensor | Mote de Contiki-NG que envía información sensada |
| Enrutador | Router 6LowPAN en Contiki-NG |
| Interfaz | Middleware que recibe datos sensados y los reeenvía a la nube |
| Nube | Azure IoT Hub notifica datos a los dispositivos suscritos |

La implementación consiste de los siguientes pasos: preparar la configuración de Azure IoT Hub utilizando llaves de acceso simétrico, desarrollar las aplicaciones para el enrutador y sensores Contiki-NG y middleware (p.e. websense). Finalmente, se comprobarán todos los programas.

El sensor ejecutará un servidor web que estará a la espera de peticiones entrantes al sensor (Ver Paso 1 y Paso 2). Mientras, la aplicación del enrutador implementará el programa $rpl-border-router" ya estudiada.

La interfaz $middleware$ solicitará datos a los sensores desde Contiki-NG y luego, los enviará al IoT Hub de Azure (Ver Pasos 3, 4 y 5).

La última aplicación se suscribirá al Azure IoT Hub para obtener los datos enviados por los sensores de Contiki-NG. Así, el programa consumirá los datos sensados que envió la interfaz $middleware$ (Ver Pasos 6, 7 y 8).

| Fuente | Referencia | 
| - | - |
| [Practical Contiki-NG](https://link.springer.com/book/10.1007/978-1-4842-3408-2) | Kurniawan, Agus. (2018). Practical Contiki-NG: Programming for Wireless Sensor Networks (p. 284). Apress. |

In [ ]:
# Paso 1 - Makefile del Sensor
CONTIKI_PROJECT = websense-Cloud
all: $(CONTIKI_PROJECT)
CONTIKI = ../.. 
PROJECT_SOURCEFILES += http-simple.c 
include $(CONTIKI)/Makefile.include

In [ ]:
# Paso 2 Adaptación Websense del Sensor
static
PT_THREAD(generate_routes(struct httpd_state *s)){
    char buff[35];
    PSOCK_BEGIN(&s->sout);
    //SEND_STRING(&s->sout, TOP);
    
    int temperature = 15 + rand() % 25;
    int humidity = 80 + rand() % 10:

    sprintf(buff,"{\"temp\":%u,\"hum\":%u}", temperature, humidity);
    printf("send json to requester\n");
    SEND_STRING(&s->sout,buff);
    //SEND_STRING(&s->sout,BOTTOM);
    PSOCK_END(&s->sout);
}

In [ ]:
# Paso 3 - Preparar Middleware con Node.js
$ npm init 
$ npm install azure-iot-device azure-iot-device-mqtt express --save
$ vi middleware-azure.js

# Paso 4 - Contenido de middleware-azure.js
'use strict';
var express = require('express');
var request = require('request');
var app = express();
var http = require('http').Server(app);
var clientFromConnectionString=require('azure-iot-device-mqtt').clientFrom-ConnectionString;
var Message = require('azure-iot-device').Message;
var connectionString = 'HostName={youriothostname};DeviceID={youriotid};SharedAccessKey={yourdevicelkey}';
var mydevideId = 'contiki-ng-01'
var client = clientFromConnection-String(connectionString);

app.get('/', function(req,res){
    res.send('Wensense Azure Cloud');
});

http.listen(3000, function(){
    console.log('listening on *:3000');
    console.log('websense azure Cloud was started');
});

function printResultFor(op){
    return function printResult(err,res){
        if(err) 
            console.log(op + 'error' + err.toString());
        if(res) 
            console.log(op + 'status' + res.constructor.name);
    };
};

var connectCallback = function(err){
     if(err) {
         console.log('Could not connect:' + err);
     } else {+
        console.log('Clinet connected');     
        setInterval(function(){
            request.get('http://[IPv6-mote-websense]/',function(err,res,body){  
                if(err){
                    console.log(err);
                    return;
                }
                var obj = JSON.parse(body);
                console.log(obj);
                var temperature = obj.temp;
                var humidity = obj.hum;
                var data = JSON.stringi-fy({devideId:mydecideId,temperature:temperature,humidity:humidity});
                var message = new Message(data);
                message.proper-ties.add('temperatureAlert',(temperature > 30)?'true':'false');
                console.log("Sending message:" + message.getData());
                client.sendEvent(message,printResultFor('send'));
            });
        },3000);
        }
};

client.open(connectCallback);
console.log('Contiki-NG Azure Middleware started.');

# Paso 5 - Comprobación middleware-azure.js
$ node middleware-azure.js

In [ ]:
# Paso 6 - Preparar Middleware con Node.js
$ npm init 
$ npm install azure-event-hubs --save
$ vi azure-sensor-subscriber.js

# Paso 7 - Contenido de azure-sensor-subscriber.js
'use strict';
var EventHubClient = require('azure-event-hubs').Client;
var connectionString = 'HostName={youriothostname},DeviceId=myiotId;SharedAccessKey={yourdevicekey}}';

var printError = function(err){
    console.log(err.message);
};

var printMessage = function(message){
    console.log('Message received:');
    console.log(JSON.stringify(message.body));
    console.log('');
};

var client = EventHubClient.fromConnectionString(connectionString);
    client.open()
        .then(client.getPartitionIds.bind(client))
        .then(function (partitionIds){
            return partitionIds.map(function(partitionId){
                return client.createReceiver('$Default',partitionId,
                                             {'startAfter-Time':Date.now()}).then(function(receiver){
                                                console.log('Created partition receiver:'+ partitionId)
                                                receiver.on('errorReceived',printError);
                                                receiver.on('message',printMessage);
                                             });                                   
            });
        })
    .catch(printError);

# Paso 8 - Comprobación azure-sensor-subscriber.js
$ node azure-sensor-subscriber.js

## Actividad 4 - Preguntas:
 
La actividad 4 implica la ejecución de Contiki-NG ya sea en un host local o en la nube de Microsoft Azure. 

| N° | Pregunta | 
| - | - |
| 4.a | ¿Cuál es el resultado presentado por la interfaz middleware? Proporcione evidencias de su implementación y describa su planteamiento. |
| 4.b | ¿Cuál es el resultado presentado por el sensor o Mote? Proporcione evidencias de su implementación y describa su planteamiento. |
| 4.c | ¿Cuál es el resultado presentado por el suscriptor de Azure Iot Hub? Proporcione evidencias de su implementación y describa su planteamiento. |

Notar: El código presentado está basado en el trabajo de Kurniawan (2018) y puede ser adaptado de acuerdo al escenario o caso de uso que determine, siendo necesario publicar dicha adaptación en la plataforma GitHub, o equivalente, para la evaluación.

# ACTIVIDAD 4 RESUELTA

### Actividad 4 — Sensores Contiki-NG en la nube

Código en `tarea4/a4-azure/` y evidencia en `tarea4/evidencia/a4-*`. **La nube de esta entrega es un broker MQTT local, no Azure.** El enunciado dice que la actividad "implica la ejecución de Contiki-NG ya sea en un host local o en la nube de Microsoft Azure" y que el código "puede ser adaptado de acuerdo al escenario o caso de uso que determine"; se tomó el host local. Mosquitto 2.0.11, que ya viene en la imagen `contiker/contiki-ng`, cumple en la cadena el mismo rol que IoT Hub: recibe la telemetría *device-to-cloud* del middleware y se la entrega al suscriptor. La cadena sensor → enrutador → middleware → broker → suscriptor corrió completa durante 90 s (`a4-azure/correr-local.sh`). El camino Azure (`azure-iot-device-mqtt` hacia IoT Hub y `@azure/event-hubs` en el suscriptor) queda como **variante implementada, no ejecutada por falta de suscripción**; los pasos para correrla están en `a4-azure/README.md` §"Azure en 5 minutos".

```
 ┌──────────────────────── contenedor contiker/contiki-ng (--privileged) ───────────────────────┐
 │  sensor/                       router/                                                      │
 │  mote Contiki-NG      6LoWPAN  rpl-border-router + tunslip6 (Cooja)   tun0 fd00::1/64        │
 │  httpd-ws :80  ──────────────▶ o TARGET=native (tun directo)  ──────▶ tcp-forward.py :8080 ──┼──┐ -p 127.0.0.1:8080
 │  {"temp":..,"hum":..}          fd00::302:304:506:708                                         │  │
 │                                                                                              │  │
 │  mosquitto :1883 (a2-mqtt/mosquitto.conf)  ◀────────────────────────────────────────────────┼──┼─┐ -p 127.0.0.1:1883
 └──────────────────────────────────────────────────────────────────────────────────────────────┘  │ │
                                                                                                   ▼ │
   middleware/  Node 22: fetch cada 3 s ──▶ {deviceId,temperature,humidity,timestamp}               │
                --destino local: mqtt (MQTT 5, QoS 1) ── usm/tel329/cloud/contiki-ng-01/telemetry ──┤
                --destino azure: azure-iot-device-mqtt ── MQTT/TLS 8883 ──▶ Azure IoT Hub (variante) │
                                                                                                     │
   subscriber/  cloud-subscriber.js: mqtt, usm/tel329/cloud/+/telemetry  ◀──────────────────────────┘
                azure-sensor-subscriber.js: @azure/event-hubs al built-in endpoint (variante Azure)
```

#### 4.b ¿Cuál es el resultado presentado por el sensor o mote?

**Planteamiento.** El sensor es un mote Contiki-NG que corre un servidor HTTP y responde a cualquier `GET` con un JSON `{"temp":T,"hum":H}`, con `T = 15 + random_rand() % 25` y `H = 80 + random_rand() % 10`, igual que el `websense-Cloud` de Kurniawan. La diferencia es cómo se consigue el servidor: Contiki-NG ya no trae `websense` ni `httpd-simple` como módulo reutilizable (sólo queda una copia privada dentro de `examples/rpl-border-router/webserver`), pero sí trae `os/net/app-layer/httpd-ws`, su descendiente directo: el proyecto define `httpd_ws_get_script()` y devuelve un protothread que escribe con `SEND_STRING`, exactamente el patrón de `generate_routes()` del enunciado.

```c
static
PT_THREAD(generate_json(struct httpd_ws_state *s))
{
  PSOCK_BEGIN(&s->sout);
  {
    unsigned temperature = 15 + (random_rand() % 25);
    unsigned humidity = 80 + (random_rand() % 10);
    snprintf(json_buf, sizeof(json_buf), "{\"temp\":%u,\"hum\":%u}", temperature, humidity);
    LOG_INFO("GET %s -> %s\n", s->filename, json_buf);
  }
  SEND_STRING(&s->sout, json_buf, strlen(json_buf));
  PSOCK_END(&s->sout);
}

httpd_ws_script_t
httpd_ws_get_script(struct httpd_ws_state *s)
{
  s->content_type = json_content_type;   /* "application/json" */
  return generate_json;
}
```

Por eso se eligió HTTP sobre CoAP: el enunciado habla de servidor web, el middleware original hace `GET`, y así el tramo Node queda con `fetch` nativo sin librerías CoAP. Hubo que habilitar TCP (`UIP_CONF_TCP 1`), fijar MSS y ventana TCP en 128 B y alinear los buffers de `httpd-ws` (sin eso no compila: la cabecera exige buffer >= ventana y GCC moderno dispara `-Werror=format-truncation` en `httpd-ws.c:255`), y forzar `Content-Type: application/json`. El mote se compiló en el contenedor para `TARGET=native`, donde la plataforma crea el `tun0` (`fd00::1/64`) y el nodo queda en `fd00::302:304:506:708`.

**Evidencia** (`evidencia/a4-sensor.log`, `evidencia/a4-curl.txt`, `evidencia/a4-sensor-local.log`). Al arrancar, el mote reporta `Routing: RPL Lite`, `Net: tun6`, link-layer `0102.0304.0506.0708` y `tun0` con `fd00::1/64`. Desde el contenedor, `ping6 -c 3 fd00::302:304:506:708` responde los tres paquetes (0 % de pérdida, RTT 0,07 a 0,77 ms) y tres `GET http://[fd00::302:304:506:708]/` devuelven `HTTP 200 OK`, `Server: Contiki`, `Content-Type: application/json` con cuerpos `{"temp":24,"hum":80}`, `{"temp":24,"hum":81}` y `{"temp":16,"hum":86}`; el mismo `GET` desde el Mac vía el puente TCP devolvió `{"temp":33,"hum":80}`. Del lado del mote, cada petición queda registrada como `[INFO: Sensor ] GET / -> {"temp":..,"hum":..}`. En la corrida de la cadena completa el mote atendió 32 `GET` (`a4-sensor-local.log`): dos del chequeo de arranque de `correr-local.sh` y 30 del middleware, uno cada 3 s, cada uno con el mismo JSON que el middleware publicó después. El enrutador `rpl-border-router` del árbol también compila para native en el contenedor (`evidencia/a4-border-router-build.log`, exit 0); la topología con Cooja + Serial Socket + `tunslip6` queda documentada en `a4-azure/router/README.md` pero no se corrió (requiere GUI).

#### 4.a ¿Cuál es el resultado presentado por la interfaz middleware?

**Planteamiento.** `middleware/middleware-azure.js` (Node 22, ESM) reemplaza al `middleware-azure.js` del libro: se quita `request` (deprecado) y Express (no participaba del flujo) y se usa `fetch` nativo con timeout. Cada 3 s consulta `SENSOR_URL`, valida que `temp` y `hum` sean números y arma `{"deviceId","temperature","humidity"}` con la propiedad `temperatureAlert = temperature > 30`. El destino se elige con `--destino local|azure` (o `DESTINO`):

- **`local` (la corrida de esta entrega).** Publica con el paquete npm `mqtt` en `MQTT_URL` (por defecto `mqtt://127.0.0.1:1883`), tema `usm/tel329/cloud/contiki-ng-01/telemetry`, QoS 1, y registra el `PUBACK` de cada mensaje con su *packet id* y el tiempo hasta recibirlo. Al cuerpo se agrega `timestamp` (ISO 8601, hora de publicación), que el suscriptor usa para medir la latencia. **Se usa MQTT 5 y `temperatureAlert` va como *user property*** (OASIS MQTT 5.0 §3.3.2.3.7): en IoT Hub la alerta es una propiedad de aplicación que viaja fuera del cuerpo, y MQTT 3.1.1 no tiene dónde ponerla salvo dentro del JSON; MQTT 5 sí, y Mosquitto 2.0 lo habla, así que el cuerpo queda como el de Azure y la alerta en el mismo lugar lógico.
- **`azure` (variante, por defecto en el código).** El SDK `azure-iot-device` + `azure-iot-device-mqtt` con su API de promesas (`Client.fromConnectionString(cs, Mqtt)`, `await client.open()`, `await client.sendEvent(message)`), la alerta como `message.properties`, la cadena de conexión sólo por entorno y el modo `--dry-run`, sin cambios respecto de la versión anterior.

```js
const client = await mqtt.connectAsync(mqttUrl, { protocolVersion: 5, clientId: `middleware-${deviceId}`, clean: true });
// por cada lectura del sensor:
const data = JSON.stringify({ deviceId, temperature, humidity, timestamp: new Date().toISOString() });
const t0 = performance.now();
const packet = await client.publishAsync(topic, data, {
  qos: 1,
  properties: { contentType: 'application/json', userProperties: { temperatureAlert: alert } },
});
console.log(`[mqtt] PUBLISH mid=${packet.messageId} ${data} temperatureAlert=${alert} -> PUBACK en ${(performance.now() - t0).toFixed(2)} ms`);
```

**Evidencia** (`evidencia/a4-middleware-local.log`, `evidencia/a4-mosquitto.log`). El middleware abrió la sesión con `Conectado a mqtt://127.0.0.1:1883 (MQTT 5) como middleware-contiki-ng-01; tema usm/tel329/cloud/contiki-ng-01/telemetry; consultando http://127.0.0.1:8080/ cada 3000 ms` y en 87 s hizo 30 lecturas del sensor y 30 `PUBLISH` QoS 1 (*packet id* 59099 a 59128), los 30 con su `PUBACK`. Cada lectura deja un par de líneas como:

```
[sensor] http://127.0.0.1:8080/ -> {"temp":32,"hum":85}
[mqtt] PUBLISH mid=59100 {"deviceId":"contiki-ng-01","temperature":32,"humidity":85,"timestamp":"2026-11-15T17:26:38.164Z"} temperatureAlert=true -> PUBACK en 1.58 ms
```

12 lecturas superaron los 30 °C y salieron con `temperatureAlert=true`; las otras 18, con `false`. Del otro lado, Mosquitto registra la misma secuencia mensaje por mensaje: `New client connected ... as middleware-contiki-ng-01 (p5, c1, k60)` (protocolo 5), y por cada dato `Received PUBLISH from middleware-contiki-ng-01 (d0, q1, r0, m59099, 'usm/tel329/cloud/contiki-ng-01/telemetry', ... (98 bytes))`, `Sending PUBLISH to cloud-subscriber-...`, `Sending PUBACK to middleware-contiki-ng-01 (m59099, rc0)` y `Received PUBACK from cloud-subscriber-...`: 30 de cada uno. El tiempo `PUBLISH → PUBACK` medido por el middleware fue de 0,73 a 11,07 ms, mediana 1,26 ms (tabla en 4.c).

#### 4.c ¿Cuál es el resultado presentado por el suscriptor de Azure IoT Hub?

**Planteamiento.** `subscriber/cloud-subscriber.js` cumple el rol de `azure-sensor-subscriber.js` con el broker local: se conecta con `mqtt` (MQTT 5) a `MQTT_URL`, se suscribe con QoS 1 a `usm/tel329/cloud/+/telemetry` (el `+` toma cualquier `deviceId`, como el suscriptor de IoT Hub recibe de todos los dispositivos del hub) e imprime, por cada mensaje, la hora de llegada, el tema, el `deviceId` sacado del tema, el cuerpo JSON, las *user properties* (`temperatureAlert`) y la latencia middleware → suscriptor, que es la hora de llegada menos el `timestamp` del cuerpo. Middleware y suscriptor corren en el mismo Mac, así que los dos lados leen el mismo reloj y no hay desfase que corregir; la resolución es de 1 ms porque `timestamp` va en ISO 8601 con milisegundos. Al salir imprime n, mínimo, mediana, media y máximo.

```js
await client.subscribeAsync('usm/tel329/cloud/+/telemetry', { qos: 1 });
client.on('message', (topic, payload, packet) => {
  const llegada = Date.now();
  const body = JSON.parse(payload.toString());
  latencias.push(llegada - Date.parse(body.timestamp));
  console.log(`Message received ${new Date(llegada).toISOString()} (topic ${topic}, deviceId=${topic.split('/')[3]}, latencia ${llegada - Date.parse(body.timestamp)} ms):`);
  console.log(payload.toString());
  console.log(`properties: ${JSON.stringify(packet.properties?.userProperties)}`);
});
```

La variante Azure, `subscriber/azure-sensor-subscriber.js`, reemplaza el paquete archivado `azure-event-hubs` por `@azure/event-hubs` 6: un `EventHubConsumerClient` con el consumer group `$Default` y la cadena "Event Hub-compatible endpoint" se suscribe a todas las particiones del endpoint integrado `messages/events` desde `latestEventPosition` (el `startAfterTime: Date.now()` del libro). Sin la cadena sale con código 1 y un mensaje que dice cuál falta (`evidencia/a4-subscriber-sinvar.log`).

**Evidencia** (`evidencia/a4-subscriber-local.log`). El suscriptor arrancó con `Suscrito a usm/tel329/cloud/+/telemetry en mqtt://127.0.0.1:1883 (MQTT 5, QoS 1); esperando mensajes...` y recibió los 30 mensajes, en orden y con el mismo cuerpo, byte a byte, que el middleware reporta como publicados. Cada uno se ve así:

```
Message received 2026-11-15T17:26:38.166Z (topic usm/tel329/cloud/contiki-ng-01/telemetry, deviceId=contiki-ng-01, latencia 2 ms):
{"deviceId":"contiki-ng-01","temperature":32,"humidity":85,"timestamp":"2026-11-15T17:26:38.164Z"}
properties: {"temperatureAlert":"true"}
```

Latencias de la corrida (90 s, un mensaje cada 3 s, middleware y suscriptor en el Mac, broker en el contenedor detrás del reenvío de puertos de OrbStack):

| tramo | n | mín | mediana | media | máx |
|---|---|---|---|---|---|
| middleware → suscriptor (llegada − `timestamp`, resolución 1 ms) | 30 | 1 ms | 1 ms | 2,2 ms | 11 ms |
| middleware `PUBLISH` → `PUBACK` (`performance.now()`) | 30 | 0,73 ms | 1,26 ms | 2,00 ms | 11,07 ms |

La mediana de 1 ms dice que entregar un dato a través del broker cuesta lo mismo que la ida y vuelta al broker: Mosquitto reenvía el `PUBLISH` al suscriptor antes de mandar el `PUBACK` al middleware (en ese orden aparecen en `a4-mosquitto.log`), así que las dos columnas miden casi el mismo camino. Las cuatro muestras de 5 ms o más (6 ms en el primer mensaje, 5, 11 y 5 ms después) coinciden una a una con los `PUBACK` lentos del middleware, o sea que son del host (planificador, reenvío de puertos de OrbStack, emulación amd64 de la imagen) y no del protocolo. Contra IoT Hub este tramo cruzaría Internet con TLS y pasaría por el endpoint compatible con Event Hubs, así que estos números son un piso del tramo nube, no una estimación de Azure.

**Publicación.** El enunciado pide publicar la adaptación en GitHub o equivalente. El código vive en el repo `dreamxist/tel329`, carpeta `tarea4/a4-azure/`, que hoy es privado: **[FALTA]** decidir si se hace público, se publica en un repo aparte o se adjunta como zip (ver `entrega/Pendientes_Tarea4.md`).

### Referencias

- Shelby, Z., Hartke, K., Bormann, C. (2014). *RFC 7252: The Constrained Application Protocol (CoAP)*. IETF. §3 (formato de mensaje), §4.2 (mensajes confirmables y retransmisión), §4.8 (parámetros de transmisión), §5 (modelo petición/respuesta), §5.2.1 (respuestas piggybacked), §6.1 (puerto 5683).
- Hartke, K. (2015). *RFC 7641: Observing Resources in CoAP*. IETF.
- OASIS (2014). *MQTT Version 3.1.1*. OASIS Standard. §3.1.2.10 (keep alive), §3.3–3.4 (PUBLISH/PUBACK), §4.2 (transporte sobre TCP), §4.3.2 (QoS 1, al menos una vez).
- OASIS (2019). *MQTT Version 5.0*. OASIS Standard. §3.3.2.3.7 (User Property en PUBLISH), §3.3.2.3.9 (Content Type).
- MQTT.js 5 (paquete npm `mqtt`), [github.com/mqttjs/MQTT.js](https://github.com/mqttjs/MQTT.js).
- TEL329, Lección 8 — pila de protocolos de la WSN (aplicación sobre UDP/uIP, IPv6 + RPL, 6LoWPAN, CSMA/802.15.4).
- TEL329, Lección 10 — CoAP: modelo de petición/respuesta y formato del mensaje.
- Contiki-NG. Documentación: [CoAP](https://docs.contiki-ng.org/en/develop/doc/programming/CoAP.html), [MQTT](https://docs.contiki-ng.org/en/develop/doc/tutorials/MQTT.html), [RPL border router](https://docs.contiki-ng.org/en/develop/doc/tutorials/RPL-border-router.html); fuentes `os/net/mac/csma/csma-output.c`, `os/net/app-layer/coap/`, `os/net/app-layer/mqtt/`, `os/net/app-layer/httpd-ws/`, `examples/coap/`, `examples/mqtt-client/`, `examples/rpl-border-router/`; `tools/cooja/doc/ChangeLog.md` ("Avoid starting the AWT thread in headless mode").
- Eclipse Mosquitto 2.0.11, [mosquitto.org](https://mosquitto.org/).
- Wireshark/TShark 3.6.2, [wireshark.org](https://www.wireshark.org/) (disectores IEEE 802.15.4, 6LoWPAN, CoAP y MQTT).
- Kurniawan, A. (2018). *Practical Contiki-NG: Programming for Wireless Sensor Networks*. Apress. Cap. "Contiki-NG en la nube".
- Microsoft Learn: [Create an IoT hub](https://learn.microsoft.com/azure/iot-hub/create-hub), [Register and connect an IoT device](https://learn.microsoft.com/azure/iot-hub/create-connect-device?tabs=portal), [Understand the built-in endpoint](https://learn.microsoft.com/azure/iot-hub/iot-hub-devguide-messages-read-builtin).
- Enunciado `TEL329-Tarea4-2026-v1.0.ipynb` (B. Contreras, UTFSM, 2026).


## Fin de la Tarea!